In [ ]:
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar
import cv2
import numpy as np

camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True
car = NvidiaRacecar()

In [ ]:
def find_lane_center(image):
    # Crop ROI — bottom portion of frame
    roi = image[150:224, :]
    
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)
    
    # Split ROI into left and right halves
    h, w = binary.shape
    mid = w // 2
    left_half = binary[:, :mid]
    right_half = binary[:, mid:]
    
    # Find centroid of white pixels in each half
    M_left = cv2.moments(left_half)
    M_right = cv2.moments(right_half)
    
    left_cx = int(M_left["m10"] / M_left["m00"]) if M_left["m00"] > 0 else None
    right_cx = int(M_right["m10"] / M_right["m00"]) + mid if M_right["m00"] > 0 else None
    
    # Compute lane center
    if left_cx is not None and right_cx is not None:
        return (left_cx + right_cx) // 2
    elif left_cx is not None:
        # Only left line visible — assume lane center is to its right
        return left_cx + 60  # estimate based on typical lane width in pixels
    elif right_cx is not None:
        return right_cx - 60
    else:
        return None

In [ ]:
def compute_error(cx, image_width=224, offset=0):
    return cx - (image_width / 2) - offset

In [ ]:
#SHREYAS CHANGE 
Kp = 0.02
Kd = 0.002

prev_error = 0

def steering_from_error(error):
    global prev_error
    derivative = error - prev_error
    prev_error = error
    steer = Kp * error + Kd * derivative
    return max(-1.0, min(1.0, steer))

In [ ]:
import time

prev_error = 0  # reset before every run

car.throttle = -0.20

try:
    while True:
        image = camera.value
        cx = find_lane_center(image)
        if cx is not None:
            error = compute_error(cx, offset= -2)
            car.steering = -steering_from_error(error)
        else:
            car.steering = 0.0
        time.sleep(0.03)
except KeyboardInterrupt:
    car.throttle = 0.0
    car.steering = 0.0

In [ ]:
import time
import cv2
import numpy as np

def find_lane_center_stable(image):
    roi = image[150:224, :]

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, binary = cv2.threshold(blur, 170, 255, cv2.THRESH_BINARY)

    h, w = binary.shape
    left_half = binary[:, :w//2]
    right_half = binary[:, w//2:]

    left_M = cv2.moments(left_half)
    right_M = cv2.moments(right_half)

    if left_M["m00"] > 0 and right_M["m00"] > 0:
        left_cx = int(left_M["m10"] / left_M["m00"])
        right_cx = int(right_M["m10"] / right_M["m00"]) + w//2
        return int((left_cx + right_cx) / 2)

    return None


def horizontal_line_score(image):
    roi = image[165:224, :]

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, binary = cv2.threshold(blur, 170, 255, cv2.THRESH_BINARY)

    row_counts = np.sum(binary == 255, axis=1)
    return np.max(row_counts)


Kp = 0.025
Kd = 0.008

prev_error = 0

def steering_from_error(error):
    global prev_error

    derivative = error - prev_error
    prev_error = error

    steer = Kp * error + Kd * derivative
    return max(-0.45, min(0.45, steer))


last_cx = None
lost_frames = 0

FRAME_CENTER = 112
BIAS = 8
target = FRAME_CENTER + BIAS

car.throttle_gain = 1
car.throttle = -0.18   # do not use -0.24 for lane correction yet

try:
    while True:
        image = camera.value

        stop_score = horizontal_line_score(image)

        if stop_score > 145:
            car.throttle = 0.0
            car.steering = 0.0
            print(f"STOP LINE detected (score={stop_score}) - stopping")
            break

        cx = find_lane_center_stable(image)

        if cx is not None:
            lost_frames = 0

            if last_cx is None:
                last_cx = cx

            else:
                old_error = abs(last_cx - target)
                new_error = abs(cx - target)

                # Accept normal movement OR any reading closer to target
                if abs(cx - last_cx) < 35 or new_error < old_error:
                    last_cx = cx
                else:
                    print(f"Ignored bad jump: old cx={last_cx}, new cx={cx}")
                    cx = last_cx

        else:
            lost_frames += 1

            if last_cx is not None and lost_frames < 3:
                cx = last_cx
                print(f"No lane detected ({lost_frames}), using last cx={last_cx}")
            else:
                car.throttle = 0.0
                car.steering = 0.0
                print("Lane lost - stopping")
                break

        error = cx - target

        # Keep current steering sign because it was smoother after removing negative
        steer = -steering_from_error(error)

        # smooth steering (prevents oscillation)
        car.steering = 0.7 * car.steering + 0.3 * steer

        print(f"cx: {cx}, target: {target}, error: {error}, steer: {steer}, stop_score: {stop_score}")

        time.sleep(0.04)

except KeyboardInterrupt:
    car.throttle = 0.0
    car.steering = 0.0
    print("Stopped manually")

In [ ]:
#trouble shooting (Stop car)
car.throttle = 0.2 # stop

In [ ]:
# trouble shooting (Parking Vision)
image = camera.value
roi = image[150:224, :]
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)

# Display the binary image
import ipywidgets
from jetcam.utils import bgr8_to_jpeg
binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
widget = ipywidgets.Image(format='jpeg')
widget.value = bgr8_to_jpeg(binary_bgr)
display(widget)

# And print the centroid
M = cv2.moments(binary)
if M["m00"] != 0:
    cx = int(M["m10"] / M["m00"])
    print(f"Centroid x: {cx} (frame center is 112)")

In [ ]:
#troubleshooting (Bay positioning)
image = camera.value
roi = image[150:224, :]
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)

h, w = binary.shape
mid = w // 2
left_half = binary[:, :mid]
right_half = binary[:, mid:]

M_left = cv2.moments(left_half)
M_right = cv2.moments(right_half)
left_cx = int(M_left["m10"] / M_left["m00"]) if M_left["m00"] > 0 else None
right_cx = int(M_right["m10"] / M_right["m00"]) + mid if M_right["m00"] > 0 else None

print(f"Left line cx: {left_cx}")
print(f"Right line cx: {right_cx}")
print(f"Lane center: {(left_cx + right_cx)//2 if left_cx and right_cx else 'N/A'}")
print(f"Frame center: 112")

In [ ]:
# Troubleshooting Simulate "lane is to the right of camera" (centroid > 112)
test_error = compute_error(150)  # positive error
print(f"Error: {test_error}")
print(f"Steering output: {-steering_from_error(test_error)}")
car.steering = -steering_from_error(test_error)
# Watch the wheels — which way do they turn?

In [ ]:
import time

# Force stop
car.throttle = 0.0
car.steering = 0.0

# Reset throttle gain
car.throttle_gain = 1

# Reset steering/throttle again
time.sleep(0.5)
car.throttle = 0.0
car.steering = 0.0

print("Car reset complete")

In [ ]:
car.throttle = 0.0
car.steering = 0.0
car.throttle_gain = 1
print("Reset complete")

In [ ]:
import time

car.throttle_gain = 1
car.steering = 0.0

car.throttle = -0.22
time.sleep(1.0)

car.throttle = 0.0
print("Forward test complete")

In [ ]:
import time
prev_error = 0

car.throttle = -0.20
try:
    while True:
        image = camera.value
        roi = image[150:224, :]
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        _, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)
        
        h, w = binary.shape
        mid = w // 2
        M_left = cv2.moments(binary[:, :mid])
        M_right = cv2.moments(binary[:, mid:])
        left_cx = int(M_left["m10"] / M_left["m00"]) if M_left["m00"] > 0 else None
        right_cx = int(M_right["m10"] / M_right["m00"]) + mid if M_right["m00"] > 0 else None
        
        if left_cx and right_cx:
            print("BOTH")
        elif left_cx:
            print("LEFT ONLY")
        elif right_cx:
            print("RIGHT ONLY")
        else:
            print("NONE")
            
        cx = find_lane_center(image)
        if cx is not None:
            error = compute_error(cx, offset=20)
            car.steering = -steering_from_error(error)
        else:
            car.steering = 0.0
        time.sleep(0.03)
except KeyboardInterrupt:
    car.throttle = 0.0
    car.steering = 0.0